<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">MARCO DE TRABAJO DE RECOMENDACIÓN DE TALENTO PARA LA FORMACIÓN DE EQUIPOS EN PROYECTOS DE INGENIERÍA DEL SOFTWARE</h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Profesores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barceló
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 5: Demostración del Framework y Ranking)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        Esta libreta culmina el desarrollo del framework <strong>NetSTAFF</strong>. Su objetivo es emular un caso de uso real donde, dados los datos de un proyecto entrante y nuestra base de empleados, un <strong>motor de machine learning explicable</strong> evalúa la idoneidad de los candidatos utilizando métricas de distancia y las reglas de conocimiento extraídas previamente.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Ingesta del Proyecto:</strong> Carga de un proyecto sintético definido mediante atributos reales (stack tecnológico, políticas de remoto, tamaño de equipo, etc.).</li>
        <li><strong>Ranking de Hard Skills (Competencias Duras):</strong> Evaluación de la idoneidad técnica midiendo la distancia del Coseno/Manhattan entre los requisitos del proyecto y las tecnologías del candidato (datos 2025).</li>
        <li><strong>Ranking de Soft Skills (Competencias Blandas):</strong> Evaluación del riesgo y la idoneidad actitudinal aplicando las reglas expertas y minadas sobre el perfil psicológico y laboral del candidato (datos 2019).</li>
        <li><strong>Justificación Metodológica:</strong> Se presentan dos rankings separados debido a la naturaleza de las fuentes de datos (años distintos), planteando su agregación ponderada como línea de trabajo futuro.</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 5.1. Configuración y Selección del Proyecto a Evaluar

Cargamos las librerías necesarias y extraemos nuestra cartera de proyectos sintéticos.

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.hard_skills import config as config_hard_skills
from src.soft_skills import config as config_soft_skills
from src.hard_skills import modelo_aux

with open(config_hard_skills.REGLAS_EXPERTAS_PATH, encoding='utf-8') as f:
    reglas_expertas = json.load(f)['reglas']

with open('../../data/raw/proyectos_sinteticos.json', encoding='utf-8') as f:
    proyectos = json.load(f)['proyectos']

reglas_soft_proyecto = [r for r in reglas_expertas if r['tipo'] == 'soft_parcial']
reglas_hard_proyecto = [r for r in reglas_expertas if r['tipo'] == 'hard_parcial']


df_candidatos_hard_skills = pd.read_csv(config_hard_skills.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId').fillna(0).astype(int)
df_candidatos_soft_skills = pd.read_csv(config_soft_skills.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId').fillna(0).astype(int)

paises_mapa = pd.read_csv(config_hard_skills.PAISES_MAPA_PATH, sep=';')


print(f"Candidatos hard skills (2025): {len(df_candidatos_hard_skills)}")
print(f"Candidatos soft skills (2019): {len(df_candidatos_soft_skills)}")
print(f"Proyectos sintéticos cargados: {len(proyectos)}")
print(f"Países cargados: {len(paises_mapa)} ")
print(f"Reglas soft_parcial evaluables con proyecto real: {len(reglas_soft_proyecto)}")
print(f"Reglas hard_parcial evaluables con proyecto real: {len(reglas_hard_proyecto)}")

Candidatos hard skills (2025): 35434
Candidatos soft skills (2019): 88778
Proyectos sintéticos cargados: 20
Países cargados: 169 
Reglas soft_parcial evaluables con proyecto real: 9
Reglas hard_parcial evaluables con proyecto real: 1


In [2]:
df_geo_hard = pd.read_csv(config_hard_skills.PROCESSED_EDA_PATH, index_col='CandidateId', usecols=['CandidateId', 'Country', 'Continente'])
df_geo_hard = df_geo_hard.merge(
    paises_mapa[['Pais_Original', 'Idiomas_Oficiales']],
    left_on='Country', right_on='Pais_Original', how='left'
)
df_geo_hard.index = pd.read_csv(config_hard_skills.PROCESSED_EDA_PATH, usecols=['CandidateId'])['CandidateId']

### 5.2. Ranking de Hard Skills — distancia Manhattan y similitud Coseno

In [3]:
def asignacion_proyecto_candidatos_hard(proyecto, df_matriz, top_n=5):
    matriz_candidatos_finales, vector_ideales, m, no_medibles = modelo_aux.vector_candidato_proyecto(
        proyecto, df_matriz, df_geo_hard, paises_mapa, config_hard_skills)
    
    pesos = modelo_aux.obtener_pesos(m, config_hard_skills) 

    # Manhattan ponderada: cada dimensión pesa distinto en el error absoluto
    distancia_manhattan = (np.abs(matriz_candidatos_finales - vector_ideales) * pesos).sum(axis=1)

    # Coseno ponderado: escalamos ambos vectores por sqrt(peso) -> producto interno ponderado
    candidato_ponderado = matriz_candidatos_finales * np.sqrt(pesos)
    ideal_ponderado = vector_ideales * np.sqrt(pesos)
    norma_candidato = np.linalg.norm(candidato_ponderado, axis=1)
    norma_ideal = np.linalg.norm(ideal_ponderado)
    with np.errstate(invalid='ignore', divide='ignore'):
        similitud_coseno = np.where(
            norma_candidato > 0,
            (candidato_ponderado @ ideal_ponderado) / (norma_candidato * norma_ideal),
            0.0,
        )

    df_ranking_final = df_matriz.copy()
    df_ranking_final['distancia_manhattan'] = distancia_manhattan
    df_ranking_final['similitud_coseno'] = similitud_coseno
    ranking_final = df_ranking_final.sort_values('distancia_manhattan').head(top_n)
    return {
        'project_id': proyecto['project_id'],
        'dimensiones_evaluadas': m,
        'no_medibles': no_medibles,
        'pesos_aplicados': dict(zip(m, pesos)),
        'ranking': ranking_final[['distancia_manhattan', 'similitud_coseno']],
    }

In [4]:
reglas_limpias_hard = pd.read_csv(config_hard_skills.REGLAS_LIMPIAS_PATH)
reglas_limpias_hard['items_totales'] = reglas_limpias_hard.apply(
    lambda row: set(str(row['antecedents']).split(' | ')) | set(str(row['consequents']).split(' | ')),
    axis=1
)
print(f"Reglas de minería cargadas: {len(reglas_limpias_hard)}")

Reglas de minería cargadas: 10107


In [5]:
def anadir_evidencia_mineria(resultado_ranking, proyecto, df_matriz, top_n_evidencia=4, max_items_evidencia=4):
    return modelo_aux.add_evidencia(resultado_ranking, proyecto, df_matriz, reglas_limpias_hard, config_hard_skills,
        top_n_evidencia=top_n_evidencia, max_items_evidencia=max_items_evidencia)

In [6]:
resultado_prueba = asignacion_proyecto_candidatos_hard(proyectos[11], df_candidatos_hard_skills)
resultado_prueba = anadir_evidencia_mineria(resultado_prueba, proyectos[11], df_candidatos_hard_skills)
modelo_aux.mostrar_ranking_proyecto(proyectos[11], resultado_prueba, df_candidatos_hard_skills, df_geo_hard, paises_mapa, config_hard_skills)

 OFERTA: Meridian Global Systems busca Backend Developer
 PROYECTO: PRY-2026-012 | Industria: Telecomunicaciones
 REQUISITOS TÉCNICOS: Java, PostgreSQL, Docker
 SENIORITY: Senior (umbral >= 0.6)
 TELETRABAJO: 100% | IDIOMA: Japanese
 PAÍSES DEL PROYECTO: Spain, Japan, Australia
 HERRAMIENTAS: Slack, Jira, Zoom
 (No evaluables por falta de mapeo/dato: os: no especificado; tools_used:Slack; tools_used:Zoom; ai_tools_preferred: no especificado)

TOP 5 CANDIDATOS:

 RANKING #1 | Candidato 26431 | Nivel de Match: 93.39% (distancia Manhattan: 3.58)
    * Ubicación: Australia | Idiomas: English
    * Cumple (14/16): Java, PostgreSQL, Docker, Nivel de seniority, % Teletrabajo, Industria, Rol / puesto, Localización, Nivel de gestión, Formación, Tipo de contrato, Herramientas de trabajo, IDE preferido, Presupuesto salarial
    * No cumple: Idioma del proyecto, Tamaño de empresa
    * Evidencia de minería (lift, sobre 3 ítems tech): sin evidencia minada
-------------------------------------------

### 5.3. Ranking de Soft Skills — riesgo activado por las reglas `soft_parcial`

### 5.4. Demostración con un proyecto real